# 05 - RQ4: Is TabNet's Attention Actually Trustworthy?

**Notebook version:** v13 -- 2026-07-29

- Train TabNet and a control MLP (same depth/width, no attention) on the reduced feature set
- Extract TabNet attention weights; use Integrated Gradients (Sundararajan et al., 2017) for the MLP control
- Compute Spearman correlation of each model's ranking against the SHAP ranking
- Compare TabNet-vs-SHAP agreement to MLP-vs-SHAP agreement to isolate attention's contribution


In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

# TODO: implement this notebook's analysis

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "05_attention_comparison_rq4"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
